In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
import json

ROOT = Path.cwd().parent if Path.cwd().name == "ml" else Path.cwd()
DATA = ROOT / "data" / "cmapss"
SPLIT = ROOT / "data" / "split.json"
TELEMETRY = ROOT / "contracts" / "telemetry.schema.json"
COLS = (["unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"]
        + [f"s{i}" for i in range(1, 22)])

def load_cmapss(path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None).iloc[:, :26]
    df.columns = COLS
    return df

train = load_cmapss(DATA / "train_FD001.txt")
print(train.shape)
train.head()

(20631, 26)


,unit_id,cycle,op_setting_1,op_setting_2,op_setting_3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [2]:
nb_moteurs = len(train["unit_id"].unique())
print(nb_moteurs)

100


In [3]:
min_cycle = train["cycle"].min()
max_cycle = train["cycle"].max()
avg_cycle = train["cycle"].mean()
print(min_cycle,max_cycle,avg_cycle)

1 362 108.80786195530997


In [4]:
train["rul"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]
print(train["rul"])

0        191
1        190
2        189
3        188
4        187
        ... 
20626      4
20627      3
20628      2
20629      1
20630      0
Name: rul, Length: 20631, dtype: int64


In [5]:
SENSORS = [f"s{i}" for i in range(1, 22)]
cv = train[SENSORS].std() / train[SENSORS].mean()
cv = cv.sort_values(ascending=False)
corr = train[SENSORS].corrwith(train["rul"])
print(corr.sort_values(ascending=False).round(3).abs())

s12    0.672
s7     0.657
s21    0.636
s20    0.629
s16    0.000
s5     0.000
s6     0.128
s14    0.307
s9     0.390
s13    0.563
s8     0.564
s3     0.585
s17    0.606
s2     0.606
s15    0.643
s4     0.679
s11    0.696
s1       NaN
s10      NaN
s18      NaN
s19      NaN
dtype: float64


In [6]:
useful_sensors = corr[corr.abs()> 0.56].index
print(useful_sensors.tolist())


['s2', 's3', 's4', 's7', 's8', 's11', 's12', 's13', 's15', 's17', 's20', 's21']


In [7]:
units = train["unit_id"].unique()
demo_units = np.random.default_rng(27).choice(units,size = 20, replace = False)
train_units = np.setdiff1d(units, demo_units)


In [8]:
with open(SPLIT, "w") as f:
    json.dump({"seed": 27,
               "demo_units": demo_units.tolist(),
               "train_units": train_units.tolist()}, f, indent=2)

In [9]:
properties = {}
for s in useful_sensors:
    lo, hi = train[s].min(), train[s].max()
    properties[s] = {
        "type": "number",
        "minimum": round(float(lo - 0.1 * abs(lo)), 4),
        "maximum": round(float(hi + 0.1 * abs(hi)), 4),
    }

schema = {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "title": "EngineTelemetry",
    "type": "object",
    "required": ["message_id", "engine_id", "cycle", "event_time", "op_settings", "sensors"],
    "additionalProperties": False,
    "properties": {
        "message_id": {"type": "string", "minLength": 8},
        "engine_id": {"type": "string", "pattern": "^ENG-[0-9]{3}$"},
        "cycle": {"type": "integer", "minimum": 1},
        "event_time": {"type": "string", "format": "date-time"},
        "op_settings": {"type": "array", "items": {"type": "number"}, "minItems": 3, "maxItems": 3},
        "sensors": {
            "type": "object",
            "required": useful_sensors.tolist(),
            "additionalProperties": {"type": "number"},
            "properties": properties,
        },
    },
}

with open(TELEMETRY, "w") as f:
    json.dump(schema, f, indent=2)

In [10]:
from uuid import uuid4
from jsonschema import Draft202012Validator, FormatChecker

validator = Draft202012Validator(json.load(open(TELEMETRY)), format_checker=FormatChecker())

def row_to_message(row):
    return {
        "message_id": str(uuid4()),
        "engine_id": f"ENG-{int(row['unit_id']):03d}",
        "cycle": int(row["cycle"]),
        "event_time": "2026-11-08T12:00:00Z",
        "op_settings": [float(row[f"op_setting_{i}"]) for i in (1, 2, 3)],
        "sensors": {s: float(row[s]) for s in useful_sensors},
    }

# 5 messages valides : 0 erreur attendue
for _, row in train.sample(5, random_state=0).iterrows():
    assert not list(validator.iter_errors(row_to_message(row)))

# cycle = -1 : doit être rejeté
bad = row_to_message(train.iloc[0]); bad["cycle"] = -1
assert list(validator.iter_errors(bad))

# capteur manquant : doit être rejeté
bad = row_to_message(train.iloc[0]); del bad["sensors"]["s2"]
assert list(validator.iter_errors(bad))